In [ ]:
import pandas as pd
import itertools
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

## generate data

In [ ]:
df_all = pd.read_csv('Data/all_data.csv', header=None)
df_all

In [ ]:
S_001_1 = df_all[1]
S_001_1.name = 'S_001_1'
S_001_2 = df_all[2]
S_001_2.name = 'S_001_2'
S_001_3 = df_all[3]
S_001_3.name = 'S_001_3'

S_002_1 = df_all[5]
S_002_1.name = 'S_002_1'
S_002_2 = df_all[6]
S_002_2.name = 'S_002_2'
S_002_3 = df_all[7]
S_002_3.name = 'S_002_3'

S_003_1 = df_all[9]
S_003_1.name = 'S_003_1'
S_003_2 = df_all[10]
S_003_2.name = 'S_003_2'
S_003_3 = df_all[11]
S_003_3.name = 'S_003_3'

S_004_1 = df_all[13]
S_004_1.name = 'S_004_1'
S_004_2 = df_all[14]
S_004_2.name = 'S_004_2'
S_004_3 = df_all[15]
S_004_3.name = 'S_004_3'

S_005_1 = df_all[17]
S_005_1.name = 'S_005_1'
S_005_2 = df_all[18]
S_005_2.name = 'S_005_2'
S_005_3 = df_all[19]
S_005_3.name = 'S_005_3'

In [ ]:
S_001 = [S_001_1, S_001_2, S_001_3]
S_002 = [S_002_1, S_002_2, S_002_3]
S_003 = [S_003_1, S_003_2, S_003_3]
S_004 = [S_004_1, S_004_2, S_004_3]
S_005 = [S_005_1, S_005_2, S_005_3]

S_list = [S_001, S_002, S_003, S_004, S_005]

s_group_conceptual_names = {
    id(S_001): "S_001_group",
    id(S_002): "S_002_group",
    id(S_003): "S_003_group",
    id(S_004): "S_004_group",
    id(S_005): "S_005_group",
}

# --- Step 2: Combine Series Groups ---
# Select all combinations of 3 Series groups from S_list
combinations_of_3_lists = list(itertools.combinations(S_list, 3))

print(f"There are a total of {len(combinations_of_3_lists)} combinations of three Series groups (5C3).")
print("=" * 70)


In [ ]:
all_resulting_dataframes = [] # Used to store all generated DataFrames

for i, group_combination_tuple in enumerate(combinations_of_3_lists):
    
    # Get the conceptual names of each Series group in the current combination for printing
    current_combo_group_names = [s_group_conceptual_names.get(id(g), "Unknown Group") for g in group_combination_tuple]
    
    series_triplets_for_df = list(itertools.product(*group_combination_tuple))

    for j, specific_series_triplet in enumerate(series_triplets_for_df):
        
        try:
            resulting_df = pd.concat(list(specific_series_triplet), axis=1)
            
            all_resulting_dataframes.append(resulting_df)
            
        except Exception as e:
            problematic_series_info = []
            for s_idx, s_val in enumerate(specific_series_triplet):
                s_name = getattr(s_val, 'name', f"Series_{s_idx}_Unnamed")
                problematic_series_info.append(str(s_name))
            print(f"    Error creating DataFrame {i+1}-{j+1}. Involved Series (original column indices): {', '.join(problematic_series_info)}.")
            print(f"    Error details: {e}")
            print("    " + "-" * 50)


print(f"\nScript execution completed. A total of {len(all_resulting_dataframes)} DataFrames were generated.")

In [ ]:
save_dir = 'Data/all_data'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
for i, df in enumerate(all_resulting_dataframes):
    columns = df.columns.tolist()
    name = '-'.join(columns)
    df_output = pd.DataFrame(columns=['Time1', '[S]1', '[catT]o1', 'Time2', '[S]2', '[catT]o2', 'Time3', '[S]3', '[catT]o3'])
    df_output['Time1'] = list(range(len(df[columns[0]].dropna()))) + [np.nan] * (len(df[columns[0]]) - len(df[columns[0]].dropna()))
    df_output['[S]1'] = df[columns[0]]
    df_output['[catT]o1'] = [float(f"0.{columns[0].split('_')[1]}")] + [np.nan] * (len(df[columns[0]]) - 1)
    df_output['Time2'] = list(range(len(df[columns[1]].dropna()))) + [np.nan] * (len(df[columns[1]]) - len(df[columns[1]].dropna()))
    df_output['[S]2'] = df[columns[1]]
    df_output['[catT]o2'] = [float(f"0.{columns[1].split('_')[1]}")] + [np.nan] * (len(df[columns[1]]) - 1)
    df_output['Time3'] = list(range(len(df[columns[2]].dropna()))) + [np.nan] * (len(df[columns[2]]) - len(df[columns[2]].dropna()))
    df_output['[S]3'] = df[columns[2]]
    df_output['[catT]o3'] = [float(f"0.{columns[2].split('_')[1]}")] + [np.nan] * (len(df[columns[2]]) - 1)
    df_output.to_csv(f'{save_dir}/{name}.csv', index=False, sep='\t')
    # break
# df_output

## predict all data

In [ ]:
# Run this notebook using the org_mech kernel.
import subprocess
import sys
subprocess.run(["bash", "predict_all_data.sh"], check=True,
               env={**os.environ, "PYTHON": sys.executable})


## analyze data

In [ ]:
dir = 'all_data_output'

df_all_results = pd.DataFrame(columns=['file_name'] + [f'M{i}' for i in range(1, 21)])

for file in os.listdir(dir):
    if file.endswith('.json'):
        with open(os.path.join(dir, file), 'r') as f:
            data = json.load(f)
        df_all_results.loc[len(df_all_results)] = [file.split('.')[0]] + data

df_all_results.to_csv('all_data_results.csv', index=False)


In [ ]:
df_all_results

In [ ]:
sns.displot(data=df_all_results, x='M2', bins=10)